# Otimização do Modelo Campeão 
### Extreme Gradient Boosting

## Biblioteca / Configuração

In [1]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

Bibliotecas instaladas


In [2]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação de dados
import pandas as pd
import numpy as np
import json
import os
import pickle
from datetime import datetime

# Visualização
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Diretórios / Funções internas
from config.paths import *
from config.function_models import *
from config.model_metrics import *

# Modelos / Machine Learning
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (roc_auc_score, precision_recall_curve, average_precision_score, roc_curve, confusion_matrix, classification_report)

# Otimização
import optuna

# Avisos
import warnings
warnings.filterwarnings("ignore")

# Configuração de exibição
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", None)

print("Ambiente Configurado")

Diretórios carregadas com sucesso
Metricas do modelo carregadas com sucesso
Ambiente Configurado


## Parâmetros Globais

In [3]:
# define a coluna alvo do modelo
TARGET = 'FPD'
# garante reprodutibilidade dos experimentos
RANDOM_STATE = 42
# Definindo o número de folds na validação cruzada
CV = 5
# data de execução do notebook (para versionamento/controle)
DATA_EXECUCAO = datetime.now().strftime('%d-%m-%Y')
# versão do pipeline/modelo
VERSAO = 'V1 - xgboost'

### Carregamento dos dados 

In [4]:
# Carregar datasets processados
train = pd.read_parquet(PROCESSED_DIR / 'abt01_train_fs.parquet')
test =  pd.read_parquet(PREDICTIONS_DIR / 'abt01_test.parquet')

print(f'Treino: {train.shape}')
print(f'Teste: {test.shape}')

Treino: (831081, 19)
Teste: (389550, 19)


# Avaliação da Performance do melhor modelo

In [5]:
# Separar features e target
X = train.drop(columns=[TARGET])
y = train[TARGET]

# Holdout de validação (para early stopping e escolha de threshold)
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Teste permanece totalmente cego para avaliação final
X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

print(f"Treino: {X_train.shape} | Validação: {X_valid.shape} | Teste: {X_test.shape}")

Treino: (664864, 18) | Validação: (166217, 18) | Teste: (389550, 18)


In [6]:
# definicao do modelo com hiperparametros padrao robustos para baseline
model = XGBClassifier(
    n_estimators=200,          # numero de arvores
    max_depth=6,               # profundidade maxima das arvores
    learning_rate=0.1,         # taxa de aprendizado
    subsample=0.8,             # amostragem de linhas por arvore
    colsample_bytree=0.8,      # amostragem de colunas por arvore
    eval_metric="logloss",     # metrica de avaliacao interna
    random_state=RANDOM_STATE,
    n_jobs=-1                  # usa todos os nucleos
)

# treino do modelo nos dados de treino
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)

### sanity check do baseline

In [7]:
# Avalia modelo baseline no conjunto de teste (AUC, PR-AUC e KS)
# probabilidades da classe positiva
probs_baseline = model.predict_proba(X_test)[:, 1]

# métricas principais
auc = roc_auc_score(y_test, probs_baseline)
pr_auc = average_precision_score(y_test, probs_baseline)

# KS
fpr, tpr, thr = roc_curve(y_test, probs_baseline)
ks = np.max(tpr - fpr)

print(f"AUC ROC (baseline): {auc:.4f}")
print(f"AUC PR  (baseline): {pr_auc:.4f}")
print(f"KS      (baseline): {ks:.4f}")

AUC ROC (baseline): 0.6991
AUC PR  (baseline): 0.3920
KS      (baseline): 0.2909


### tratar desbalanceamento

In [8]:
# Treina XGBoost nativo com balanceamento de classe e early stopping

# calcula peso da classe positiva automaticamente
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Scale_pos_weight: {scale_pos_weight:.2f}")

# cria estrutura nativa do XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)
dtest  = xgb.DMatrix(X_test, label=y_test)

# parâmetros robustos para crédito
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'eta': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'seed': RANDOM_STATE,
    'nthread': -1
}

# treino com early stopping em validação
model_xgb = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,
    evals=[(dvalid, 'valid')],
    early_stopping_rounds=50,
    verbose_eval=False
)

print(f"Best iteration: {model_xgb.best_iteration}")

# probabilidades no teste
probs_balanced = model_xgb.predict(dtest)

Scale_pos_weight: 3.31
Best iteration: 256


In [9]:
# Avalia modelo com balanceamento

auc = roc_auc_score(y_test, probs_balanced)
pr_auc = average_precision_score(y_test, probs_balanced)

fpr, tpr, thr = roc_curve(y_test, probs_balanced)
ks = np.max(tpr - fpr)

print(f"AUC ROC (balanced): {auc:.4f}")
print(f"AUC PR  (balanced): {pr_auc:.4f}")
print(f"KS      (balanced): {ks:.4f}")

AUC ROC (balanced): 0.6995
AUC PR  (balanced): 0.3928
KS      (balanced): 0.2913


### Optuna

In [10]:
# Otimiza hiperparâmetros do XGBoost com Optuna usando validação cruzada estratificada
# (sem usar o conjunto de teste)

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'eta': trial.suggest_float('eta', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0, 8),
        'lambda': trial.suggest_float('lambda', 0.1, 8, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 8, log=True),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
        'seed': RANDOM_STATE,
        'nthread': -1
    }

    skf = StratifiedKFold(n_splits=CV, shuffle=True, random_state=RANDOM_STATE)
    aucs = []

    for train_idx, valid_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

        neg_fold = (y_tr == 0).sum()
        pos_fold = (y_tr == 1).sum()
        params['scale_pos_weight'] = neg_fold / pos_fold

        dtrain_fold = xgb.DMatrix(X_tr, label=y_tr)
        dvalid_fold = xgb.DMatrix(X_va, label=y_va)

        model = xgb.train(
            params=params,
            dtrain=dtrain_fold,
            num_boost_round=1200,
            evals=[(dvalid_fold, 'valid')],
            early_stopping_rounds=100,
            verbose_eval=False
        )

        preds = model.predict(dvalid_fold)
        aucs.append(roc_auc_score(y_va, preds))

    return float(np.mean(aucs))

# cria estudo bayesiano
study = optuna.create_study(direction="maximize")

# roda busca
study.optimize(objective, n_trials=80, show_progress_bar=True)

print("\nMelhor AUC média CV:", study.best_value)
print("Melhores parâmetros encontrados:")
print(study.best_params)

[I 2026-03-05 19:08:30,655] A new study created in memory with name: no-name-f1cb0c9a-36eb-41fa-bbe3-d8275cd71b9d


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-05 19:10:23,483] Trial 0 finished with value: 0.7074171515988953 and parameters: {'max_depth': 4, 'eta': 0.146492743548953, 'subsample': 0.7148080154494437, 'colsample_bytree': 0.8110099079407069, 'min_child_weight': 14, 'gamma': 4.812670547228961, 'lambda': 0.28990584786714446, 'alpha': 0.24853348709691717, 'max_delta_step': 3}. Best is trial 0 with value: 0.7074171515988953.
[I 2026-03-05 19:11:37,958] Trial 1 finished with value: 0.7060907845896911 and parameters: {'max_depth': 6, 'eta': 0.1397718205909289, 'subsample': 0.6516085382641951, 'colsample_bytree': 0.6008095073145127, 'min_child_weight': 10, 'gamma': 2.1595963353448955, 'lambda': 0.3038154092865033, 'alpha': 0.0012760959245936108, 'max_delta_step': 9}. Best is trial 0 with value: 0.7074171515988953.
[I 2026-03-05 19:18:14,565] Trial 2 finished with value: 0.707015009250723 and parameters: {'max_depth': 3, 'eta': 0.015065409888614793, 'subsample': 0.6052748875450517, 'colsample_bytree': 0.6855040914538586, 'min_

### Treinando o melhores parâmetros

In [11]:
# Treina modelo final com melhores hiperparâmetros encontrados no Optuna

# peso da classe positiva no treino
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

# cria DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)
dtest  = xgb.DMatrix(X_test, label=y_test)

# parâmetros finais vindos do Optuna + ajustes fixos
params_final = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'scale_pos_weight': scale_pos_weight,
    'seed': RANDOM_STATE,
    'nthread': -1,
    **study.best_params
}

# treino com early stopping em validação
model_final = xgb.train(
    params=params_final,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=[(dvalid, 'valid')],
    early_stopping_rounds=100,
    verbose_eval=False
)

print(f"Best iteration final: {model_final.best_iteration}")

# probabilidades finais no teste (teste continua cego no treino)
probs_final = model_final.predict(dtest)

Best iteration final: 1134


In [12]:
# Avalia modelo final

auc = roc_auc_score(y_test, probs_final)
pr_auc = average_precision_score(y_test, probs_final)

fpr, tpr, thr = roc_curve(y_test, probs_final)
ks = np.max(tpr - fpr)

print(f"AUC ROC (final): {auc:.4f}")
print(f"AUC PR  (final): {pr_auc:.4f}")
print(f"KS      (final): {ks:.4f}")

AUC ROC (final): 0.6995
AUC PR  (final): 0.3929
KS      (final): 0.2918


In [13]:
# Calcula KS e métricas por threshold usando o modelo final
from sklearn.metrics import roc_curve, precision_score, recall_score, f1_score, confusion_matrix

# calcula KS
fpr, tpr, thresholds = roc_curve(y_test, probs_final)
ks_values = tpr - fpr
best_idx = np.argmax(ks_values)

best_threshold = thresholds[best_idx]
best_ks = ks_values[best_idx]

print(f"KS máximo: {best_ks:.4f}")
print(f"Threshold KS: {best_threshold:.4f}")

# tabela operacional de decisão
grid = np.linspace(0.05, 0.80, 16)

rows = []
for thr in grid:
    preds = (probs_final >= thr).astype(int)
    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    f1   = f1_score(y_test, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    rows.append([thr, prec, rec, f1, tp, fp, tn, fn])

metrics_table = pd.DataFrame(
    rows,
    columns=["threshold","precision","recall","f1","TP","FP","TN","FN"]
)

display(metrics_table.sort_values("threshold"))

KS máximo: 0.2918
Threshold KS: 0.4894


,threshold,precision,recall,f1,TP,FP,TN,FN
0,0.05,0.226534,0.999943,0.369385,88224,301228,93,5
1,0.10,0.230267,0.997155,0.374137,87978,294091,7230,251
2,0.15,0.237640,0.988768,0.383186,87238,279863,21458,991
3,0.20,0.247233,0.973376,0.394313,85880,261484,39837,2349
4,0.25,0.258835,0.949336,0.406766,83759,239841,61480,4470
5,0.30,0.272326,0.912784,0.419497,80534,215192,86129,7695
6,0.35,0.287375,0.867300,0.431706,76521,189755,111566,11708
7,0.40,0.305095,0.807206,0.442820,71219,162213,139108,17010
8,0.45,0.324449,0.732752,0.449755,64650,134611,166710,23579
9,0.50,0.348576,0.641399,0.451681,56590,105756,195565,31639


### Salvando o modelo

In [14]:
# salva o modelo nativo XGBoost
model_final.save_model(MODELS_DIR / "modelo_credito_xgb.json")

In [15]:
# Carregar JSON (mais portátil)
model_json = xgb.Booster()
model_json.load_model(MODELS_DIR / "modelo_credito_xgb.json")